# 🦥 Unsloth LoRA Studio: Merge, Quantize & Publish

**Description:** interactive notebook to merge LoRA adapters with base LLMs, quantize them into multiple GGUF formats simultaneously, and push them to Hugging Face. Intened for use in Google Colab.

### ⚡ Features
* **Interactive Dashboard:** No more editing code variables. Use dropdowns and buttons.
* **Multi-Quantization:** Select multiple formats (e.g., Q4_K_M, Q8_0) and process them in one go.
* **Smart Upload:** Drag-and-drop your LoRA zip file (should include config.json and adapter weights) and upload the merged & quantized model to Hugging Face with a single click.

### 🚀 Getting Started
1. Set your Hugging Face Write Token in colab or .env or paste it manually.
2. Upload your LoRA zip file using the drag-and-drop widget.
3. Select your base model from the dropdown (adapter has to be compatible with the base model).
4. Choose desired quantization formats.
5. Click "Merge & Quantize" to start the process.

In [ ]:
# @title 1. Initialize Environment
# @markdown Run this cell to install Unsloth and prepare the dashboard dependencies.
import torch
import os

try:
    import unsloth
except ImportError:
    # Check GPU to choose the correct Unsloth version
    major_version, minor_version = torch.cuda.get_device_capability()
    if major_version >= 8:
        !pip install "unsloth[colab-ampere] @ git+https://github.com/unslothai/unsloth.git" --quiet
    else:
        !pip install "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git" --quiet

    !pip install --no-deps "trl<0.8.6" peft accelerate bitsandbytes --quiet
    !pip install llama-cpp-python --quiet
    !pip install huggingface_hub --quiet

# Import the necessary libraries for the next cell
import ipywidgets as widgets
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print(f"✅ Custom widget manager enabled (ipywidgets {widgets.__version__}).")
except Exception as widget_err:
    print(f"⚠️ Could not enable custom widget manager automatically: {widget_err}")
from IPython.display import display, clear_output
import shutil
import zipfile
from huggingface_hub import HfApi, login
from unsloth import FastLanguageModel
from google.colab import userdata
print("✅ Environment Ready. Run the next cell to open the Studio.")

In [ ]:
from dotenv import load_dotenv
import os
from google.colab import userdata
from google.colab.userdata import SecretNotFoundError

# @title 2. 🦁 Unsloth Studio
# @markdown Run this cell to launch the dashboard.

# ==========================================
# 1. SETUP & STYLING
# ==========================================
style = {'description_width': '120px'}
layout = widgets.Layout(width='98%')

header = widgets.HTML("<h2>🦁 Unsloth Studio: LoRA Merger</h2>")

# Load HF token from Colab secrets, .env, or fallback to manual entry
hf_token_value = ""
hf_token_source = "manual entry"

# 1. Try Colab Secrets
try:
    colab_secret_token = userdata.get('HF_TOKEN')
    if colab_secret_token:
        hf_token_value = colab_secret_token
        hf_token_source = "Colab Secret"
except SecretNotFoundError:
    pass
except Exception as e:
    print(f"Warning: An unexpected error occurred while loading Colab Secret: {e}")
    pass

# 2. If not found in Colab Secrets, try .env file
if not hf_token_value:
    try:
        load_dotenv()
        env_token = os.environ.get("HF_TOKEN")
        if env_token:
            hf_token_value = env_token
            hf_token_source = ".env file"
    except ImportError:
        pass

hf_token_widget = widgets.Password(
    placeholder=f'Paste your HF Write Token here' if not hf_token_value else f'Loaded from {hf_token_source}',
    description='<b>HF Token:</b>',
    value=hf_token_value,
    style=style,
    layout=layout
)

# ==========================================
# 2. DYNAMIC MODEL SEARCH (FUZZY)
# ==========================================
from huggingface_hub import HfApi

def fetch_top_models():
    print("⏳ Connecting to Hugging Face to fetch top models...")
    try:
        api = HfApi()
        models = api.list_models(filter="text-generation", sort="downloads", direction="-1", limit=50)
        return [m.id for m in models]
    except Exception:
        print("⚠️ Search failed. Using fallback list.")
        return ["Qwen/Qwen3-8B", "Qwen/Qwen3-32B"]

try:
    model_options = fetch_top_models()
    clear_output()
except Exception:
    model_options = ["Qwen/Qwen3-8B", "Qwen/Qwen3-32B"]

# Fuzzy search widgets
search_model_text = widgets.Text(
    placeholder='Type to fuzzy search (e.g. qwen8b)',
    description='<b>Search:</b>',
    style=style,
    layout=layout,
)
model_dropdown = widgets.Dropdown(
    options=model_options,
    value=model_options[0] if model_options else None,
    description='<b>Base Model:</b>',
    style=style,
    layout=layout,
)

# Install / import rapidfuzz for fuzzy matching
try:
    from rapidfuzz import process, fuzz
except ImportError:
    !pip install rapidfuzz --quiet
    from rapidfuzz import process, fuzz

FUZ_LIMIT = 15

def fuzzy_update(query):
    if not query:
        model_dropdown.options = model_options
        return
    try:
        matches = process.extract(query, model_options, scorer=fuzz.WRatio, limit=FUZ_LIMIT)
        model_dropdown.options = [m[0] for m in matches]
        if model_dropdown.value not in model_dropdown.options and model_dropdown.options:
            model_dropdown.value = model_dropdown.options[0]
    except Exception:
        model_dropdown.options = model_options

def on_search_change(change):
    if change['name'] == 'value':
        fuzzy_update(change['new'].strip())

search_model_text.observe(on_search_change, names='value')

# ==========================================
# 3. UPLOAD & CONFIG
# ==========================================
upload_widget = widgets.FileUpload(
    accept='.zip',
    multiple=False,
    description='Upload LoRA .zip',
    layout=widgets.Layout(width='30%')
)
upload_status = widgets.Label(value="📁 Waiting for file...")

def _entry_to_file(entry):
    """Convert FileUpload entries (dict or UploadedFile) into plain dicts."""
    if isinstance(entry, dict):
        return {
            'name': entry.get('name') or entry.get('metadata', {}).get('name', 'Unknown'),
            'content': entry.get('content'),
        }
    name = getattr(entry, 'name', None) or getattr(getattr(entry, 'metadata', {}), 'get', lambda *_: None)('name')
    content = getattr(entry, 'content', None)
    if content is None and hasattr(entry, 'data'):
        content = entry.data
    return {'name': name or 'Unknown', 'content': content}

def extract_uploaded_files(upload_value):
    """Return a normalized list of uploaded files (name + content)."""
    if not upload_value:
        return []
    files = []
    if isinstance(upload_value, dict):
        for key, entry in upload_value.items():
            normalized = _entry_to_file(entry)
            if normalized.get('name') == 'Unknown':
                normalized['name'] = key
            files.append(normalized)
    elif isinstance(upload_value, (tuple, list)):
        for entry in upload_value:
            files.append(_entry_to_file(entry))
    else:
        # ipywidgets<8 stored content directly on .value via traitlets Bytes
        maybe_content = getattr(upload_widget, 'data', None)
        if maybe_content:
            files.append({'name': 'adapter.zip', 'content': maybe_content[0]})
    return [f for f in files if f.get('content') not in (None, b'', memoryview(b''))]

def on_upload_change(change):
    if change['name'] != 'value':
        return
    try:
        files = extract_uploaded_files(change['new'])
        if files:
            upload_status.value = f"✅ Uploaded: {files[0]['name']}"
        else:
            upload_status.value = "📁 Waiting for file..."
    except Exception as upload_err:
        upload_status.value = f"⚠️ Upload error: {upload_err}"

upload_widget.observe(on_upload_change, names='value')

# Full Quantization Options
quant_options = [
    "f16", "bf16", "q8_0",
    "q6_k",
    "q5_k_l", "q5_k_m", "q5_k_s", "q5_0", "q5_1",
    "q4_k_l", "q4_k_m", "q4_k_s", "q4_0", "q4_1", "iq4_nl", "iq4_xs",
    "q3_k_l", "q3_k_m", "q3_k_s", "iq3_xxs",
    "q2_k", "iq2_xxs", "iq2_xs", "iq2_s", "iq2_m",
    "iq1_s", "iq1_m"
]

preselected = {'q4_k_m', 'q8_0'}
quant_checkboxes = {opt: widgets.Checkbox(value=opt in preselected, description=opt, indent=False) for opt in quant_options}

row_size = 6
rows = []
current = []
for idx, opt in enumerate(quant_options, 1):
    current.append(quant_checkboxes[opt])
    if idx % row_size == 0 or idx == len(quant_options):
        rows.append(widgets.HBox(current))
        current = []

quant_toggle_btn = widgets.Button(
    description='Select All',
    button_style='info',
    icon='check-square',
    layout=widgets.Layout(width='150px')
)

selected_count_label = widgets.Label()

def update_quant_button_label():
    total = len(quant_checkboxes)
    selected = sum(cb.value for cb in quant_checkboxes.values())
    quant_toggle_btn.description = 'Deselect All' if total and selected == total else 'Select All'
    selected_count_label.value = f"Selected: {selected}/{total}"

def on_quant_checkbox_change(change):
    if change['name'] == 'value':
        update_quant_button_label()

for cb in quant_checkboxes.values():
    cb.observe(on_quant_checkbox_change, names='value')

update_quant_button_label()

def set_all_quant_checkboxes(target_value):
    for cb in quant_checkboxes.values():
        cb.unobserve(on_quant_checkbox_change, names='value')
        cb.value = target_value
        cb.observe(on_quant_checkbox_change, names='value')

def on_quant_toggle_clicked(_):
    all_selected = all(cb.value for cb in quant_checkboxes.values()) if quant_checkboxes else False
    set_all_quant_checkboxes(not all_selected)
    update_quant_button_label()

quant_toggle_btn.on_click(on_quant_toggle_clicked)

quant_container = widgets.VBox([
    widgets.HBox([widgets.HTML('<b>Formats:</b>'), quant_toggle_btn, selected_count_label]),
    *rows
])

username_widget = widgets.Text(placeholder='Your HF Username', description='<b>Username:</b>', style=style, layout=layout)
modelname_widget = widgets.Text(value='my-finetune-gguf', description='<b>New Model Name:</b>', style=style, layout=layout)

run_btn = widgets.Button(
    description='🚀 MERGE & UPLOAD',
    button_style='success',
    layout=widgets.Layout(width='100%', height='60px'),
    icon='bolt'
)

log_output = widgets.Output(layout={'border': '1px solid #444', 'height': '300px', 'overflow_y': 'scroll', 'padding': '10px'})

# ==========================================
# 4. EXECUTION LOGIC
# ==========================================

def log(message):
    with log_output:
        print(message)

def get_selected_quant_methods():
    return [opt for opt, cb in quant_checkboxes.items() if cb.value]

def execute_pipeline(_):
    log_output.clear_output()
    try:
        log("🔍 Starting validation...")
        log(f"Token present: {bool(hf_token_widget.value)}")
        upload_val = upload_widget.value
        upload_repr = list(upload_val.keys()) if isinstance(upload_val, dict) else upload_val
        log(f"Upload widget raw value: {upload_repr}")

        if not hf_token_widget.value:
            log("❌ ERROR: Missing Hugging Face Token.")
            return

        uploaded_files = extract_uploaded_files(upload_widget.value)
        if not uploaded_files:
            log("❌ ERROR: Missing LoRA .zip file. Please upload a file first.")
            log("💡 TIP: Try re-uploading the file after the cell finishes executing.")
            return

        primary_file = uploaded_files[0]
        fname = primary_file.get('name', 'adapter.zip')
        content = primary_file.get('content')
        if isinstance(content, memoryview):
            content = content.tobytes()
        if not content:
            log("❌ ERROR: Uploaded file content is empty.")
            return

        log(f"✓ Upload detected: {fname}")

        log("🔑 Connecting to Hugging Face...")
        login(token=hf_token_widget.value)

        log("📂 Processing LoRA file...")
        upload_dir = "lora_upload"
        if os.path.exists(upload_dir):
            shutil.rmtree(upload_dir)
        os.makedirs(upload_dir)

        zip_path = os.path.join(upload_dir, "adapter.zip")
        with open(zip_path, "wb") as f:
            f.write(content)

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(upload_dir)

        lora_path = upload_dir
        config_found = False
        for root, dirs, files in os.walk(upload_dir):
            if "adapter_config.json" in files:
                lora_path = root
                config_found = True
                break

        if not config_found:
            log("❌ ERROR: adapter_config.json not found in zip! Is this a valid LoRA?")
            return

        log(f"✅ LoRA detected at: {lora_path}")

        model_id = model_dropdown.value
        log(f"⏳ Loading Base Model: {model_id}...")

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = model_id,
            max_seq_length = 2048,
            dtype = None,
            load_in_4bit = True,
        )

        log("🔗 Reading adapter_config.json & Attaching LoRA...")
        model.load_adapter(lora_path)
        log("✅ Model Merged successfully (Parameters auto-detected).")

        repo_id = f"{username_widget.value}/{modelname_widget.value}"
        methods = get_selected_quant_methods()

        log(f"🚀 TARGET REPO: https://huggingface.co/{repo_id}")
        if not methods:
            log("⚠️ No quantization formats selected. Nothing to upload.")
            return

        for method in methods:
            log(f"\n⚙️ Processing Format: {method} ...")
            if method in ["f16", "bf16"]:
                model.push_to_hub_merged(
                    repo_id,
                    tokenizer,
                    save_method = "merged_16bit",
                    token = hf_token_widget.value,
                )
            else:
                model.push_to_hub_gguf(
                    repo_id,
                    tokenizer,
                    quantization_method = method,
                    token = hf_token_widget.value,
                )
            log(f"✅ {method} Uploaded!")

        log("\n✨ ALL DONE! ✨")

    except Exception as e:
        log(f"\n❌ FATAL ERROR: {str(e)}")
        import traceback
        log(traceback.format_exc())

run_btn.on_click(execute_pipeline)

# ==========================================
# 5. RENDER UI
# ==========================================
ui = widgets.VBox([
    header, hf_token_widget, widgets.HTML("<hr>"),
    widgets.HBox([search_model_text, model_dropdown]),
    widgets.HBox([upload_widget, upload_status]),
    widgets.HTML("<hr>"), widgets.HBox([username_widget, modelname_widget]),
    quant_container, widgets.HTML("<br>"), run_btn,
    widgets.HTML("<h4>📜 Execution Log</h4>"), log_output
])

display(ui)
